In [ ]:
# Exploratory queries against the built read model.
#
# Read-only. Build it first with scripts/build_panchayat_db.py or
# notebooks/build_database.ipynb.

import duckdb
from database import config

con = duckdb.connect(str(config.DB_PATH), read_only=True)

def q(sql):
    return con.execute(sql).df()

q("SHOW TABLES")


In [ ]:
# How many activities did each gram panchayat plan, and what did they budget?
q("""
SELECT g.gp_name, a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned_cost
FROM planned_activity a
JOIN gram_panchayat g USING (gp_lgd_code)
GROUP BY 1, 2 ORDER BY 1, 2
""")


In [ ]:
# Planned versus spent, and the utilisation rate.
#
# Expenditure is one-to-many on activity_code, so it must be aggregated to one
# row per activity BEFORE the join. Joining first repeats total_cost per
# expenditure row and inflates both the activity count and the planned total.
q("""
SELECT a.fiscal_year,
       count(*) AS activities,
       sum(a.total_cost) AS planned,
       sum(e.total_expenditure) AS spent,
       round(100.0 * sum(e.total_expenditure) / nullif(sum(a.total_cost), 0), 1) AS util_pct
FROM planned_activity a
LEFT JOIN (
    SELECT activity_code, sum(total_expenditure) AS total_expenditure
    FROM activity_expenditure
    GROUP BY activity_code
) e USING (activity_code)
GROUP BY 1 ORDER BY 1
""")


In [ ]:
# Trace an activity from plan through approval to the vouchers that paid it.
q("""
SELECT a.activity_code, a.activity_name, a.total_cost,
       aa.adm_approval_no, ta.tec_approval_cost,
       av.voucher_no, av.voucher_cost, v.date
FROM planned_activity a
LEFT JOIN admin_approval     aa USING (activity_code)
LEFT JOIN technical_approval ta USING (activity_code)
LEFT JOIN activity_expenditure e USING (activity_code)
LEFT JOIN activity_voucher   av USING (expenditure_id)
LEFT JOIN voucher            v  USING (voucher_pk)
ORDER BY av.voucher_cost DESC NULLS LAST
LIMIT 15
""")


In [ ]:
# Decode a coded column through dim_code.
q("""
SELECT d.description AS focus_area_name, count(*) AS activities
FROM planned_activity a
JOIN dim_code d
  ON d.variable = 'focus_area'
 AND d.code = CAST(a.focus_area AS VARCHAR)
GROUP BY 1 ORDER BY 2 DESC
""")


In [ ]:
con.close()
